# 03b — APTOS Fine-Tuning (Stage 2)

Fine-tune the EyePACS pre-trained EfficientNetV2-S on the APTOS 2019 dataset.

**Why two stages?**
- EyePACS has ~35K images but noisy labels → learns robust retinal features
- APTOS has ~3.6K images with clean clinical labels → calibrates grade boundaries

**Strategy:**
- Load best Stage 1 checkpoint (`best_model_s1_phase3.pth`)
- Full backbone unfreeze with very low LR (1e-5 backbone, 1e-4 head)
- EMA for stable weight averaging
- Evaluate on APTOS test set


In [ ]:
import importlib
import os
import sys

IS_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/diabetic_retinopathy"
    %pip install -q timm albumentations kaggle scipy
else:
    PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import src.config
importlib.reload(src.config)
from src import config
config.ensure_dirs()


In [ ]:
import torch
import pandas as pd

from src.augmentations import get_train_transforms, get_valid_transforms
from src.data_manager import ensure_aptos_dataset
from src.dataset import (
    compute_class_weights,
    create_dataloaders,
    load_and_split_dataframe,
    preprocess_images_to_cache,
)
from src.evaluate import compute_metrics, plot_training_curves, predict
from src.model import (
    ModelEMA,
    build_model,
    count_parameters,
    unfreeze_backbone,
)
from src.trainer import Trainer
from src.utils import get_device, log_gpu_memory, seed_everything

seed_everything()
device = get_device()
log_gpu_memory("Initial")


## 1. Download & Prepare APTOS Dataset


In [ ]:
# Download APTOS if not present (auto-downloads from Kaggle)
ensure_aptos_dataset()

# Load and split
train_df, valid_df, test_df = load_and_split_dataframe(dataset_name="aptos")

print(f"Train : {len(train_df):,} images")
print(f"Valid : {len(valid_df):,} images")
print(f"Test  : {len(test_df):,} images")
print("\nTrain class distribution:")
print(train_df["level"].value_counts().sort_index())


## 2. Preprocessing Cache + DataLoaders


In [ ]:
# Build/verify preprocessed cache
print("Building APTOS preprocessed cache...")
preprocess_images_to_cache(train_df, dataset_name="aptos")
preprocess_images_to_cache(valid_df, dataset_name="aptos")
preprocess_images_to_cache(test_df, dataset_name="aptos")
print("Cache ready.")


In [ ]:
# Create dataloaders
train_tfm = get_train_transforms(config.IMG_SIZE)
valid_tfm = get_valid_transforms(config.IMG_SIZE)

loaders = create_dataloaders(
    train_df=train_df,
    valid_df=valid_df,
    dataset_name="aptos",
    test_df=test_df,
    train_transform=train_tfm,
    valid_transform=valid_tfm,
    batch_size=config.STAGE2_BATCH_SIZE,
)

# Class weights
class_weights = compute_class_weights(train_df["level"].values)
print("Class weights:")
for name, w in zip(config.CLASS_NAMES, class_weights):
    print(f"  {name:20s}: {w:.4f}")

print(f"\nTrain batches: {len(loaders['train'])}")
print(f"Valid batches: {len(loaders['valid'])}")
print(f"Test batches:  {len(loaders['test'])}")


## 3. Load Stage 1 Checkpoint


In [ ]:
# Build model and load pretrained Stage 1 weights
model = build_model(pretrained=False)

s1_ckpt = os.path.join(config.CHECKPOINTS_DIR, "best_model_s1_phase3.pth")
if os.path.isfile(s1_ckpt):
    ckpt = torch.load(s1_ckpt, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    print(f"Loaded Stage 1 checkpoint: {s1_ckpt}")
    print(f"  Stage 1 QWK: {ckpt.get('qwk', 'N/A')}")
else:
    print(f"WARNING: Stage 1 checkpoint not found at {s1_ckpt}")
    print("Using ImageNet pretrained weights instead.")
    model = build_model(pretrained=True)

# Full unfreeze for fine-tuning
unfreeze_backbone(model, from_block=None)
print(f"\nParameters: {count_parameters(model)}")

model = model.to(device)


## 4. Fine-Tune on APTOS

| Setting | Value |
|---------|-------|
| LR (backbone) | 1e-5 |
| LR (head) | 1e-4 (10x) |
| Weight decay | 5e-4 |
| Epochs | 20 |
| Mixup | Yes |
| EMA | Yes |


In [ ]:
# EMA + Trainer
ema = ModelEMA(model) if config.USE_EMA else None

trainer = Trainer(
    model=model,
    device=device,
    class_weights=class_weights,
    ema=ema,
)

log_gpu_memory("After model init")

# Fine-tune
trainer.fit_phase(
    phase="aptos_finetune",
    train_loader=loaders["train"],
    valid_loader=loaders["valid"],
    epochs=config.STAGE2_EPOCHS,
    lr=config.STAGE2_LR,
    weight_decay=config.STAGE2_WD,
    warmup_epochs=2,
    use_mixup=config.USE_MIXUP,
    use_amp=True,
)


## 5. Training Curves


In [ ]:
plot_training_curves(
    trainer.history,
    save_path=os.path.join(config.FIGURES_DIR, "stage2_training_curves.png"),
)
trainer.save_history(os.path.join(config.LOGS_DIR, "stage2_history.json"))


## 6. Evaluate on APTOS Test Set


In [ ]:
# Load best checkpoint
trainer.load_best_checkpoint("aptos_finetune")

if ema is not None:
    ema.apply(model)
    print("Applied EMA weights for evaluation")

# Predict on test set
y_true, y_pred, y_proba = predict(model, loaders["test"], device)
metrics = compute_metrics(y_true, y_pred, y_proba)

print(f"APTOS Test QWK      : {metrics['qwk']:.4f}")
print(f"APTOS Test Accuracy : {metrics['accuracy']:.4f}")
print(f"APTOS Test ROC-AUC  : {metrics['roc_auc_weighted']:.4f}")
print(f"\n{metrics['classification_report_str']}")


## 7. Threshold Optimization

Standard argmax may not be optimal for ordinal classes.
We optimize thresholds on the validation set to maximize QWK.


In [ ]:
from scipy.optimize import minimize

def optimize_thresholds(y_true, y_proba, n_classes=5):
    """Find optimal thresholds that maximize QWK on validation set."""
    from sklearn.metrics import cohen_kappa_score
    
    # Use continuous predictions (expected value)
    y_continuous = (y_proba * torch.arange(n_classes).float()).sum(dim=1).numpy() \
        if isinstance(y_proba, torch.Tensor) else (y_proba * range(n_classes)).sum(axis=1)
    
    # Initial thresholds at class boundaries
    init_thresholds = [0.5, 1.5, 2.5, 3.5]
    
    def neg_qwk(thresholds):
        t = sorted(thresholds)
        import numpy as np
        preds = np.digitize(y_continuous, t)
        return -cohen_kappa_score(y_true, preds, weights="quadratic")
    
    result = minimize(neg_qwk, init_thresholds, method="Nelder-Mead")
    optimal_thresholds = sorted(result.x)
    
    import numpy as np
    y_pred_opt = np.digitize(y_continuous, optimal_thresholds)
    qwk_opt = cohen_kappa_score(y_true, y_pred_opt, weights="quadratic")
    
    return optimal_thresholds, qwk_opt, y_pred_opt

# Optimize on validation set
y_true_val, y_pred_val, y_proba_val = predict(model, loaders["valid"], device)
opt_thresholds, qwk_opt, _ = optimize_thresholds(y_true_val, y_proba_val)

# Apply to test set
import numpy as np
y_continuous_test = (y_proba * np.arange(5)).sum(axis=1) if not isinstance(y_proba, torch.Tensor) \
    else (y_proba * torch.arange(5).float()).sum(dim=1).numpy()
y_pred_opt_test = np.digitize(y_continuous_test, opt_thresholds)

from sklearn.metrics import cohen_kappa_score
qwk_argmax = cohen_kappa_score(y_true, y_pred, weights="quadratic")
qwk_threshold = cohen_kappa_score(y_true, y_pred_opt_test, weights="quadratic")

print(f"Optimal thresholds: {[f'{t:.3f}' for t in opt_thresholds]}")
print(f"\nTest QWK (argmax):              {qwk_argmax:.4f}")
print(f"Test QWK (optimized thresholds): {qwk_threshold:.4f}")
print(f"Improvement:                     {qwk_threshold - qwk_argmax:+.4f}")


## 8. ONNX Export


In [ ]:
from src.export import export_to_onnx

onnx_path = export_to_onnx(
    checkpoint_path=os.path.join(config.CHECKPOINTS_DIR, "best_model_aptos_finetune.pth"),
)
print(f"\nONNX model exported to: {onnx_path}")


## 9. Save Test Split

Save for use in `04_evaluation.ipynb`.


In [ ]:
test_csv_path = os.path.join(config.OUTPUTS_DIR, "test_split_aptos.csv")
test_df.to_csv(test_csv_path, index=False)
print(f"APTOS test split saved: {test_csv_path} ({len(test_df)} images)")
